# Stage 3 · iLLaDA-8B-Instruct with thinking off: answer prefix, slots and scoring-code tests

Runtime: **L4 or A100**. One fresh runtime for this notebook only. iLLaDA runs on LLaDA's official code (`mask_id=5`) and is scored unbatched, so test 3 checks repeat runs only.

The first run (`results/stage3/illada/`) failed the usable-scores gate: iLLaDA opens every answer with a `<think>` block, so its first answer slot holds `<`, not Yes/No. As recorded in `decisions/2026-10-25-illada-thinking-off.md`, this run turns thinking off with an empty think block after the generation prompt, choosing the exact text and the slot count on the development bank by median Yes+No mass. Results go to `results/stage3/illada-thinking-off/`.

In [ ]:
BRANCH = "main"   # the branch to test; main once the stage-3 PR is merged
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone -q -b {BRANCH} https://github.com/Manas-33/CSCI-544-Course-Project.git /content/project
%cd /content/project
!git log --oneline -1
!pip install -q -r harness/diffusion/requirements/illada.txt
!git clone -q https://github.com/ML-GSAI/LLaDA /content/LLaDA && git -C /content/LLaDA checkout -q 9182493
OUT = "results/stage3/illada-thinking-off"
!mkdir -p {OUT}
import hashlib
DEV_HASH = hashlib.sha256(open("bank/dev/dev_bank.jsonl", "rb").read()).hexdigest()
SANITY_HASH = hashlib.sha256(open("bank/sanity/sanity_set.jsonl", "rb").read()).hexdigest()
print(DEV_HASH, SANITY_HASH)

## 1. Choose the answer prefix and slot count on the development bank
Scores the 142 development answers with every answer prefix and 1, 8 and 32 slots (labels are never read), then picks the combination with the highest median Yes+No mass; ties go to fewer slots, then to the simpler prefix. Also reports the usable-scores gate. The rule is fixed in `decisions/2026-10-25-illada-thinking-off.md`.

In [ ]:
for p in ("none", "think_nl", "think_nlnl"):
    for n in (1, 8, 32):
        !python -m harness.diffusion.run_score --model illada --answer-prefix {p} --n-slots {n} --items bank/dev/dev_bank.jsonl --bank-hash {DEV_HASH} --out {OUT}/dev_T1_{p}_n{n}.jsonl
FILES = " ".join(f"{OUT}/dev_T1_{p}_n{n}.jsonl" for p in ("none", "think_nl", "think_nlnl") for n in (1, 8, 32))
!python -m harness.diffusion.choose_slots {FILES} --report {OUT}/slots.json
import json
choice = json.load(open(f"{OUT}/slots.json"))
N, PREFIX = choice["chosen_n_slots"], choice["chosen_answer_prefix"]
print("answer slots:", N, "| answer prefix:", PREFIX)

## 2. Smoke test on a tiny random-weight copy of the official code (CPU)

In [ ]:
!python -m harness.diffusion.checks.check_tiny --model illada --llada-repo /content/LLaDA --answer-prefix {PREFIX} --report {OUT}/tiny.json

## 3. Scoring-code tests 1–3 (PLAN.md §4)
Test 2 compares our scorer with the official model code's first forward pass; test 1 is the sanity bar (AUROC ≥ 0.90); test 3 checks batched against unbatched scores and repeat runs.

In [ ]:
!python -m harness.diffusion.checks.check_parity --model illada --n-slots {N} --answer-prefix {PREFIX} --llada-repo /content/LLaDA --report {OUT}/test2_parity_n{N}.json
!python -m harness.diffusion.checks.check_sanity --model illada --n-slots {N} --answer-prefix {PREFIX} --report {OUT}/test1_sanity_n{N}.json --out {OUT}/sanity_scores_n{N}.jsonl
!python -m harness.diffusion.checks.check_invariance --model illada --n-slots {N} --answer-prefix {PREFIX} --report {OUT}/test3_invariance_n{N}.json --save {OUT}/invariance_scores_n{N}.json

Each check prints PASS or FAIL and writes its report even when it fails; a FAIL is a result, not a crash. Keep going.

## Download the results
This zips `results/stage3/illada-thinking-off/` (scores, manifests and reports). Send the zip back, or unzip it into the repo at the same path.

In [ ]:
!cd /content/project && zip -qr /content/stage3_illada-thinking-off.zip results/stage3/illada-thinking-off
from google.colab import files
files.download("/content/stage3_illada-thinking-off.zip")